In [1]:
import torch.nn as nn
import torch.utils.data.dataloader
from torch.nn.functional import scaled_dot_product_attention
import numpy as np
from src.language_models.utils import repackage_hidden, get_batch, batchify, save_checkpoint, move_to_device, save_val_loss_data
from src.language_models.dictionary_corpus import Corpus
from tqdm import tqdm

## Model

In [48]:
class SRNN_Softmax (nn.Module):
    def __init__(self, ntokens,nhid,ninp, device, n_layers=1, memory_size=104, memory_dim = 5):#replaced output_size parameter by ntokens
        super(SRNN_Softmax, self).__init__()
        self.ntokens = ntokens
        self.ninp = ninp
        self.n_layers = n_layers
        self.nhid = nhid
        self.device= device
        
        self.memory_size = memory_size
        self.memory_dim = memory_dim
        
        self.rnn = nn.RNN(self.ninp, self.nhid, self.n_layers)

        self.W_y = nn.Linear(self.nhid, ntokens)
        self.W_n = nn.Linear(self.nhid, self.memory_dim)
        self.W_a = nn.Linear(self.nhid, 2)
        self.W_sh = nn.Linear (self.memory_dim, self.nhid)
        self.encoder = nn.Embedding(self.ntokens, self.ninp)
        self.decoder = nn.Linear(nhid, ntokens)
        print(ntokens)
        print(ninp)
        # Actions -- push : 0 and pop: 1
        self.softmax = nn.Softmax(dim=2) 
        self.sigmoid = nn.Sigmoid ()
        self.softmax_out = nn.Softmax(dim=-1)
    
    def init_hidden (self, batch_size):
        return torch.zeros (self.n_layers, batch_size, self.nhid).to(self.device)#additional modification to allow batch processing
    
    def forward(self, input, hidden0, stack, temperature=1.):
        print('input', input.shape)
        emb = self.encoder(input)
        print('emb', emb.shape)
        print('hidden0', hidden0.shape)
        print('stack[0]', stack[0].shape)
        print('self.W_sh (stack[0])', self.W_sh (stack[0]).shape)
        hidden_bar = self.W_sh (stack[0]).view(1, 1, -1) + hidden0
        print('hidden_bar',hidden_bar.shape)
        ht, hidden = self.rnn(emb, hidden_bar)
        print('ht',ht.shape)
        print('hidden', hidden.shape)
        print('self.sigmoid(self.W_y(ht))', self.sigmoid(self.W_y(ht)).shape)
        print('self.sigmoid(self.W_y(ht)).view(-1, self.ntokens)', self.sigmoid(self.W_y(ht)).view(-1, self.ntokens).shape)
        print('self.decoder(self.sigmoid(self.W_y(ht)).view(-1, self.ntokens))', self.decoder(self.sigmoid(self.W_y(ht))).shape)
        output = self.softmax_out(self.decoder(self.sigmoid(self.W_y(ht)).view(-1, self.ntokens)))
        print('output', output.shape)
        self.action_weights = self.softmax (self.W_a (ht)).view(-1)
        print('action_weights', self.action_weights.shape)
        self.new_elt = self.sigmoid (self.W_n(ht)).view(1, self.memory_dim)
        print('self.new_elt',self.new_elt.shape)
        push_side = torch.cat ((self.new_elt, stack[:-1]), dim=0)
        print('push_side',push_side.shape)
        pop_side = torch.cat ((stack[1:], torch.zeros(1, self.memory_dim).to(self.device)), dim=0)
        print('pop_side',pop_side.shape)
        stack = self.action_weights [0] * push_side + self.action_weights [1] * pop_side
        print('stack', stack.shape)
        return output, hidden, stack

## Training function

In [6]:
def train(model, criterion, train_data, batch_size, ntokens, memory_size, memory_dim, device, nheads=False):
    # Turn on training mode which enables dropout.
    model.train()
    total_loss = 0
    #NEW : move hidden to devide
    
    
    for batch, i in enumerate(tqdm(range(0, train_data.size(0) - 1, 35), desc="Training")):
        data, targets = get_batch(train_data, i, 35)
        #NEW : move data and target to device
        model.zero_grad()
        hidden = model.init_hidden(batch_size)
        memory = torch.zeros (memory_size, memory_dim).to(device)        # truncated BPP
        
        output, hidden, memory = model(data, hidden,memory)
        print(output.shape)
        break
        # output_flat = output.reshape(-1, output.size(-1))
        
        # # Similarly, reshape targets to [seq_len*batch_size]
        # targets_flat = targets.reshape(-1)
        # #loss = criterion(output.view(-1, ntokens), targets)
        # loss=criterion(output_flat, targets_flat)
        # loss.backward()

        # # `clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
        # torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
        # for p in model.parameters():
        #     p.data.add_(-10, p.grad.data)

        # total_loss += loss.item()

## Data

In [3]:
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')
ntokens = len(corpus.dictionary)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_data = batchify(corpus.train, 128, device)#batch size of 128
val_data = batchify(corpus.valid, 128, device)
test_data = batchify(corpus.test, 128, device)
criterion = nn.CrossEntropyLoss()


In [18]:
ntokens

50001

In [49]:
model = SRNN_Softmax(ntokens, 128, 128, device)

50001
128


In [50]:
train(model, criterion, train_data, 128, ntokens, 104, 5, device)#memory size and memory dim by default in the github


Training:   0%|          | 0/18540 [00:00<?, ?it/s]

input torch.Size([35, 128])
emb torch.Size([35, 128, 128])
hidden0 torch.Size([1, 128, 128])
stack[0] torch.Size([5])
self.W_sh (stack[0]) torch.Size([128])
hidden_bar torch.Size([1, 128, 128])
ht torch.Size([35, 128, 128])
hidden torch.Size([1, 128, 128])
self.sigmoid(self.W_y(ht)) torch.Size([35, 128, 50001])


Training:   0%|          | 0/18540 [00:00<?, ?it/s]

self.sigmoid(self.W_y(ht)).view(-1, self.ntokens) torch.Size([4480, 50001])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (4480x50001 and 128x50001)